In [1]:
from google.colab import drive
drive.mount('/content/drive')
from google.colab import userdata

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# Task 3.1 - Compute & Report Metrics

## Calculate Metrics

In [2]:
!pip install -q ragas==0.4.3 datasets openai langchain-openai

In [3]:
# ============================================================
# TASK 3 — END-TO-END RAGAS EVALUATION
# INPUT:   task_3_0_generation__section_chunks__baseline.json  (or any results JSON with same schema)
# OUTPUT:  task_3_1_evaluation__section_chunks__baseline.json
# METRICS: FAITHFULNESS | ANSWER RELEVANCY | CONTEXT PRECISION
#          CONTEXT RECALL | ANSWER CORRECTNESS | HIT RATE@3
# MODEL:   GPT-4O-MINI (RAGAS LLM JUDGE) + TEXT-EMBEDDING-3-SMALL
# ============================================================

# ── INSTALL DEPENDENCIES ─────────────────────────────────────

import os
import json
import time
import math
import numpy as np
from pathlib import Path
from collections import defaultdict

from datasets import Dataset
from ragas import evaluate
from ragas.metrics import (
    faithfulness,
    answer_relevancy,
    context_precision,
    context_recall,
    answer_correctness,
)  # **RAGAS 0.4+ IMPORT PATH — AVOIDS DEPRECATION WARNINGS**

from langchain_openai import ChatOpenAI, OpenAIEmbeddings  # **LLM + EMBEDDINGS FOR RAGAS**

print("✅ All imports successful")

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


✅ All imports successful


/tmp/ipykernel_59883/1861122335.py:22: DeprecationWarning: Importing faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import faithfulness
  from ragas.metrics import (
/tmp/ipykernel_59883/1861122335.py:22: DeprecationWarning: Importing answer_relevancy from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import answer_relevancy
  from ragas.metrics import (
/tmp/ipykernel_59883/1861122335.py:22: DeprecationWarning: Importing context_precision from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import context_precision
  from ragas.metrics import (
/tmp/ipykernel_59883/1861122335.py:22: DeprecationWarning: Importing context_recall from 'ragas.metrics' is deprecated and will be

In [4]:
# ══════════════════════════════════════════════════════════
# CONFIG — EDIT PATHS AND API KEY HERE
# ══════════════════════════════════════════════════════════

# ── FILE PATHS ────────────────────────────────────────────
RESULTS_PATH = "/content/drive/MyDrive/Assignment3/task_3_0_generation__section_chunks__baseline.json"   # **INPUT: GENERATION RESULTS JSON**
QRELS_PATH   = "/content/drive/MyDrive/Assignment3/qrels.json"             # **QRELS FOR HIT RATE GROUND TRUTH**
OUTPUT_PATH  = "/content/drive/MyDrive/Assignment3/task_3_1_evaluation__section_chunks__baseline.json"  # **OUTPUT: PER-PAIR + AGGREGATE METRICS**

# ── OPENAI CONFIG ─────────────────────────────────────────
OPENAI_API_KEY = userdata.get('OPENAI_API_KEY')         # **REPLACE WITH YOUR OPENAI API KEY**
os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')
RAGAS_LLM      = "gpt-4o-mini"           # **LLM USED BY RAGAS AS JUDGE**
RAGAS_EMBED    = "text-embedding-3-small"# **EMBEDDING MODEL USED BY RAGAS**

# ── EVALUATION SETTINGS ───────────────────────────────────
HIT_K          = 3           # **HIT RATE@K — K=3 PER ASSIGNMENT SPEC**
BATCH_SIZE     = 25          # **RAGAS BATCH SIZE PER API CALL**
LIMIT          = None        # **SET TO INT (e.g. 50) FOR QUICK TESTING; None = ALL 496**


print(f"Config loaded | HIT_K={HIT_K} | BATCH={BATCH_SIZE} | LIMIT={LIMIT}")

Config loaded | HIT_K=3 | BATCH=25 | LIMIT=None


In [5]:
# ══════════════════════════════════════════════════════════
# STEP 1 — LOAD DATA
# ══════════════════════════════════════════════════════════

print("Loading results JSON...")
with open(RESULTS_PATH, "r") as f:
    raw = json.load(f)  # **LOAD FULL RESULTS FILE**

results = raw["results"]  # **EXTRACT RESULT RECORDS**
if LIMIT:
    results = results[:LIMIT]  # **OPTIONAL SLICE FOR QUICK TESTING**

print(f"Loading qrels JSON...")
with open(QRELS_PATH, "r") as f:
    qrels = json.load(f)  # **GROUND TRUTH DOC IDS PER QUERY**

print(f"✅ Loaded {len(results)} results | {len(qrels)} qrels")
print(f"   Metadata — task: {raw.get('task')} | model: {raw.get('model')} | "
      f"strategy: {raw.get('strategy')} | collection: {raw.get('collection')}")

Loading results JSON...
Loading qrels JSON...
✅ Loaded 496 results | 496 qrels
   Metadata — task: 3.evaluation | model: gpt-4o-mini | strategy: baseline | collection: section_chunks


In [6]:
# ══════════════════════════════════════════════════════════
# STEP 2 — VALIDATE SCHEMA
# ══════════════════════════════════════════════════════════

REQUIRED_FIELDS = {"query_id", "query", "answer", "ground_truth",
                   "modality", "contexts", "retrieved_chunks"}  # **EXPECTED KEYS**

sample = results[0]
missing = REQUIRED_FIELDS - set(sample.keys())
if missing:
    raise ValueError(f"❌ Results JSON is missing required fields: {missing}")
else:
    print("✅ Schema validation passed")

# SHOW FIRST RECORD SUMMARY **
print(f"\nSample record:")
print(f"  query_id   : {sample['query_id']}")
print(f"  modality   : {sample['modality']}")
print(f"  query      : {sample['query'][:80]}")
print(f"  answer     : {sample['answer'][:100]}...")
print(f"  ground_truth: {sample['ground_truth'][:100]}...")
print(f"  contexts   : {len(sample['contexts'])} chunks")
print(f"  chunks     : {[c['arxiv_id'] for c in sample['retrieved_chunks']]}")

# MODALITY BREAKDOWN **
from collections import Counter
modality_counts = Counter(r["modality"] for r in results)
print(f"\nModality breakdown: {dict(modality_counts)}")

✅ Schema validation passed

Sample record:
  query_id   : 1d585069-a446-47fa-a74d-0387316ea330
  modality   : text-table
  query      : In what areas do syllabic embeddings show potential for improvement based on cur
  answer     : I don't have enough information in the provided context to answer this question. 

Sources Used: Non...
  ground_truth: Syllabic embeddings could be improved in areas such as speaker identity detection, slot filling, and...
  contexts   : 3 chunks
  chunks     : ['2410.07168v2', '2406.02166v2', '2410.07168v2']

Modality breakdown: {'text-table': 28, 'text-image': 115, 'text': 329, 'text-table-image': 24}


In [7]:
# ══════════════════════════════════════════════════════════
# STEP 3 — HIT RATE@K (NO API NEEDED)
# ══════════════════════════════════════════════════════════

def compute_hit_rate_from_results(
    results: list,
    qrels: dict,
    k: int = HIT_K
) -> tuple:
    """
    COMPUTE HIT RATE@K FROM PRE-COMPUTED RESULTS JSON**
    A HIT = CORRECT ARXIV_ID APPEARS IN TOP-K RETRIEVED CHUNKS**
    RETURNS: (OVERALL_HIT_RATE, PER_PAIR_HIT_LIST, FAILURES)**
    """
    per_pair_hits = []   # **ONE BOOL PER Q&A PAIR**
    failures      = []   # **LOG MISSED RETRIEVALS FOR ERROR ANALYSIS**
    skipped       = 0    # **QUERIES WITHOUT QRELS ENTRY**

    for rec in results:
        qid = rec["query_id"]

        if qid not in qrels:
            # SKIP QUERIES WITHOUT GROUND TRUTH MAPPING **
            skipped += 1
            per_pair_hits.append(None)
            continue

        ground_truth_doc = qrels[qid]["doc_id"]  # **CORRECT ARXIV ID FROM QRELS**

        # EXTRACT TOP-K ARXIV IDS FROM retrieved_chunks **
        top_k_ids = [
            c["arxiv_id"]
            for c in rec["retrieved_chunks"][:k]  # **SLICE TO K**
        ]

        hit = ground_truth_doc in top_k_ids  # **TRUE IF CORRECT PAPER IN TOP K**
        per_pair_hits.append(hit)

        if not hit:
            failures.append({
                "query_id":     qid,
                "query":        rec["query"],
                "modality":     rec["modality"],
                "ground_truth": ground_truth_doc,
                "retrieved":    top_k_ids,
            })  # **LOG FOR ERROR ANALYSIS**

    valid_hits = [h for h in per_pair_hits if h is not None]
    hit_rate   = sum(valid_hits) / len(valid_hits) if valid_hits else 0.0

    if skipped:
        print(f"  ⚠️ Skipped {skipped} queries with no qrels entry")

    return round(hit_rate, 4), per_pair_hits, failures


print(f"Computing Hit Rate@{HIT_K}...")
overall_hit_rate, per_pair_hits, hr_failures = compute_hit_rate_from_results(
    results, qrels, k=HIT_K
)

valid_hits = [h for h in per_pair_hits if h is not None]
print(f"\n{'═'*50}")
print(f"  Hit Rate@{HIT_K}  : {overall_hit_rate:.4f}  "
      f"({sum(valid_hits)}/{len(valid_hits)} queries)")
print(f"  Failures      : {len(hr_failures)}")
print(f"{'═'*50}")

Computing Hit Rate@3...

══════════════════════════════════════════════════
  Hit Rate@3  : 0.9677  (480/496 queries)
  Failures      : 16
══════════════════════════════════════════════════


In [8]:
# ══════════════════════════════════════════════════════════
# STEP 4 — HIT RATE@K BREAKDOWN BY MODALITY
# ══════════════════════════════════════════════════════════

def hit_rate_by_modality(
    results: list,
    per_pair_hits: list,
    qrels: dict
) -> dict:
    """
    BREAK DOWN HIT RATE BY QUERY MODALITY**
    MODALITIES: text | text-image | text-table | text-table-image**
    """
    modality_hits  = defaultdict(list)

    for rec, hit in zip(results, per_pair_hits):
        if hit is None:
            continue  # **SKIP QUERIES WITHOUT QRELS**
        modality = rec["modality"]
        modality_hits[modality].append(hit)

    breakdown = {}
    for mod, hits in sorted(modality_hits.items()):
        breakdown[mod] = {
            "hit_rate": round(sum(hits) / len(hits), 4),
            "hits":     sum(hits),
            "total":    len(hits),
        }  # **AGGREGATE PER MODALITY**

    return breakdown


modality_hr = hit_rate_by_modality(results, per_pair_hits, qrels)

print(f"Hit Rate@{HIT_K} by modality:")
print(f"  {'Modality':<20} {'Hit Rate':>10} {'Hits':>6} {'Total':>7}")
print(f"  {'-'*45}")
for mod, stats in modality_hr.items():
    print(f"  {mod:<20} {stats['hit_rate']:>10.4f} "
          f"{stats['hits']:>6} {stats['total']:>7}")

Hit Rate@3 by modality:
  Modality               Hit Rate   Hits   Total
  ---------------------------------------------
  text                     0.9666    318     329
  text-image               0.9826    113     115
  text-table               0.9286     26      28
  text-table-image         0.9583     23      24


In [9]:
# ══════════════════════════════════════════════════════════
# STEP 5 — CONFIGURE RAGAS LLM + EMBEDDINGS
# ══════════════════════════════════════════════════════════
# ROOT CAUSE OF embed_text + NLIStatementOutput ERRORS:
# RAGAS 0.4.x REQUIRES ITS OWN WRAPPER CLASSES AROUND
# LANGCHAIN OBJECTS — PASSING RAW ChatOpenAI / OpenAIEmbeddings
# DIRECTLY SKIPS THE embed_text() + JSON-CLEANING ADAPTERS **

import os
os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY  # **SET API KEY FOR RAGAS**

from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from ragas.llms   import LangchainLLMWrapper        # **RAGAS ADAPTER FOR LLMs**
from ragas.embeddings import LangchainEmbeddingsWrapper  # **RAGAS ADAPTER FOR EMBEDDINGS**

# STEP 1: BUILD RAW LANGCHAIN OBJECTS **
_lc_llm = ChatOpenAI(
    model=RAGAS_LLM,
    temperature=0.0,
    openai_api_key=OPENAI_API_KEY,
)  # **RAW LANGCHAIN LLM — NOT PASSED TO RAGAS DIRECTLY**

_lc_embeddings = OpenAIEmbeddings(
    model=RAGAS_EMBED,
    openai_api_key=OPENAI_API_KEY,
)  # **RAW LANGCHAIN EMBEDDINGS — NOT PASSED TO RAGAS DIRECTLY**

# STEP 2: WRAP WITH RAGAS ADAPTERS **
# LangchainLLMWrapper  — ADDS JSON FENCE STRIPPING SO NLIStatementOutput PARSES CLEANLY **
# LangchainEmbeddingsWrapper — EXPOSES embed_text() METHOD THAT RAGAS METRICS CALL **
ragas_llm        = LangchainLLMWrapper(_lc_llm)
ragas_embeddings = LangchainEmbeddingsWrapper(_lc_embeddings)

# STEP 3: INJECT INTO EVERY METRIC **
METRICS = [
    faithfulness,
    answer_relevancy,
    context_precision,
    context_recall,
    answer_correctness,
]
for metric in METRICS:
    metric.llm        = ragas_llm         # **ALL METRICS NEED THE LLM WRAPPER**
    metric.embeddings = ragas_embeddings  # **answer_relevancy USES EMBEDDINGS FOR COSINE SIM**

print(f"✅ RAGAS configured | LLM: {RAGAS_LLM} | Embeddings: {RAGAS_EMBED}")
print(f"   LLM wrapper    : {type(ragas_llm).__name__}")
print(f"   Embed wrapper  : {type(ragas_embeddings).__name__}")
print(f"   Metrics        : {[m.name for m in METRICS]}")

✅ RAGAS configured | LLM: gpt-4o-mini | Embeddings: text-embedding-3-small
   LLM wrapper    : LangchainLLMWrapper
   Embed wrapper  : LangchainEmbeddingsWrapper
   Metrics        : ['faithfulness', 'answer_relevancy', 'context_precision', 'context_recall', 'answer_correctness']


/tmp/ipykernel_59883/3141853745.py:31: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use llm_factory instead: from openai import OpenAI; from ragas.llms import llm_factory; llm = llm_factory('gpt-4o-mini', client=OpenAI(api_key='...'))
  ragas_llm        = LangchainLLMWrapper(_lc_llm)
/tmp/ipykernel_59883/3141853745.py:32: DeprecationWarning: LangchainEmbeddingsWrapper is deprecated and will be removed in a future version. Use the modern embedding providers instead: embedding_factory('openai', model='text-embedding-3-small', client=openai_client) or from ragas.embeddings import OpenAIEmbeddings, GoogleEmbeddings, HuggingFaceEmbeddings
  ragas_embeddings = LangchainEmbeddingsWrapper(_lc_embeddings)


In [10]:
# ══════════════════════════════════════════════════════════
# STEP 6 — BUILD RAGAS DATASET
# ══════════════════════════════════════════════════════════

def build_ragas_dataset(results: list) -> Dataset:
    """
    CONVERT RESULTS LIST TO A RAGAS-COMPATIBLE HuggingFace DATASET**
    RAGAS REQUIRES: question | answer | contexts | ground_truth**
    CONTEXTS MUST BE A LIST OF STRINGS (ONE PER RETRIEVED CHUNK)**
    """
    records = []
    for rec in results:
        answer_text = rec["answer"]
        # SKIP FAILED GENERATIONS — THEY SKEW ALL METRICS DOWN **
        if not answer_text or answer_text.strip() in (
            "Generation failed.",
            "Generation failed due to API error.",
            ""
        ):
            continue

        records.append({
            "question":     rec["query"],
            "answer":       answer_text,
            "contexts":     rec["contexts"],        # **LIST[STR] — RETRIEVED CHUNKS**
            "ground_truth": rec["ground_truth"],    # **REFERENCE ANSWER FROM answers.json**
        })

    skipped = len(results) - len(records)
    if skipped:
        print(f"  ⚠️ Skipped {skipped} failed generations before building dataset")

    return Dataset.from_list(records), records  # **RETURN BOTH FOR INDEX ALIGNMENT**


print("Building RAGAS dataset...")
ragas_dataset, ragas_records = build_ragas_dataset(results)
print(f"✅ Dataset ready | {len(ragas_dataset)} records")
print(f"   Columns: {ragas_dataset.column_names}")

Building RAGAS dataset...
✅ Dataset ready | 496 records
   Columns: ['question', 'answer', 'contexts', 'ground_truth']


In [11]:
# ══════════════════════════════════════════════════════════
# STEP 7 — RUN RAGAS EVALUATION IN BATCHES (WITH CHECKPOINTS)
# ══════════════════════════════════════════════════════════

import os
from pathlib import Path

# Create checkpoint directory
CHECKPOINT_DIR = Path("/content/drive/MyDrive/Assignment3/checkpoint_task_3_1_evaluation__section_chunks__baseline")
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

def run_ragas_batched(
    dataset: Dataset,
    metrics: list,
    batch_size: int = BATCH_SIZE,
    checkpoint_dir: Path = CHECKPOINT_DIR
) -> list:
    """
    RUN RAGAS EVALUATE IN BATCHES TO AVOID API RATE LIMITS
    SAVES CHECKPOINT AFTER EACH BATCH FOR RECOVERY
    RETURNS A LIST OF PER-SAMPLE SCORE DICTS (ONE PER RECORD)
    """
    all_scores   = []
    n            = len(dataset)
    n_batches    = math.ceil(n / batch_size)

    print(f"Running RAGAS on {n} records | {n_batches} batches of {batch_size}")
    print(f"Checkpoints will be saved to: {checkpoint_dir}/")

    for i in range(n_batches):
        start = i * batch_size
        end   = min(start + batch_size, n)
        batch = dataset.select(range(start, end))

        print(f"  Batch {i+1}/{n_batches}  (rows {start}–{end-1})...", end=" ", flush=True)

        batch_scores = []  # Scores for this batch only
        batch_success = False

        for attempt in range(3):
            try:
                result = evaluate(
                    dataset=batch,
                    metrics=metrics,
                    raise_exceptions=False,
                )
                batch_df = result.to_pandas()

                for _, row in batch_df.iterrows():
                    batch_scores.append({
                        "faithfulness":      row.get("faithfulness",      None),
                        "answer_relevancy":  row.get("answer_relevancy",  None),
                        "context_precision": row.get("context_precision", None),
                        "context_recall":    row.get("context_recall",    None),
                        "answer_correctness":row.get("answer_correctness",None),
                    })

                batch_success = True
                print(f"✅  ({end - start} records)")
                break

            except Exception as e:
                print(f"\n    ⚠️ Attempt {attempt+1}/3 failed: {e}")
                if attempt < 2:
                    time.sleep(5)
                else:
                    for _ in range(end - start):
                        batch_scores.append({
                            "faithfulness":       None,
                            "answer_relevancy":   None,
                            "context_precision":  None,
                            "context_recall":     None,
                            "answer_correctness": None,
                        })
                    print(f"    ❌ Batch {i+1} failed — NaN inserted for {end-start} rows")

        # Add batch scores to cumulative list
        all_scores.extend(batch_scores)

        # ── SAVE BATCH CHECKPOINT ─────────────────────────────
        checkpoint = {
            "batch_number": i + 1,
            "total_batches": n_batches,
            "row_start": start,
            "row_end": end,
            "batch_size": batch_size,
            "batch_success": batch_success,
            "timestamp": time.strftime("%Y-%m-%d %H:%M:%S"),
            "scores": batch_scores,
        }

        checkpoint_path = checkpoint_dir / f"batch_{i+1:04d}.json"
        with open(checkpoint_path, "w") as f:
            json.dump(checkpoint, f, indent=2, default=str)

        # Also save cumulative progress
        cumulative_path = checkpoint_dir / "cumulative_progress.json"
        cumulative = {
            "last_completed_batch": i + 1,
            "total_batches": n_batches,
            "total_rows_processed": len(all_scores),
            "total_rows": n,
            "timestamp": time.strftime("%Y-%m-%d %H:%M:%S"),
            "all_scores": all_scores,
        }
        with open(cumulative_path, "w") as f:
            json.dump(cumulative, f, indent=2, default=str)

        print(f"       💾 Checkpoint saved: {checkpoint_path.name}")

        # Rate limit buffer between batches
        if i < n_batches - 1:
            time.sleep(1)

    print(f"\n📁 All checkpoints saved in: {checkpoint_dir}/")
    return all_scores


print("Starting RAGAS evaluation...")
ragas_scores = run_ragas_batched(ragas_dataset, METRICS, batch_size=BATCH_SIZE)
print(f"\n✅ RAGAS evaluation complete | {len(ragas_scores)} records scored")

Starting RAGAS evaluation...
Running RAGAS on 496 records | 20 batches of 25
Checkpoints will be saved to: /content/drive/MyDrive/Assignment3/checkpoint_task_3_1_evaluation__section_chunks__baseline/
  Batch 1/20  (rows 0–24)... 

Evaluating:   0%|          | 0/125 [00:00<?, ?it/s]

✅  (25 records)
       💾 Checkpoint saved: batch_0001.json
  Batch 2/20  (rows 25–49)... 

Evaluating:   0%|          | 0/125 [00:00<?, ?it/s]

✅  (25 records)
       💾 Checkpoint saved: batch_0002.json
  Batch 3/20  (rows 50–74)... 

...

✅  (25 records)
       💾 Checkpoint saved: batch_0018.json
  Batch 19/20  (rows 450–474)... 

Evaluating:   0%|          | 0/125 [00:00<?, ?it/s]

✅  (25 records)
       💾 Checkpoint saved: batch_0019.json
  Batch 20/20  (rows 475–495)... 

Evaluating:   0%|          | 0/105 [00:00<?, ?it/s]

✅  (21 records)
       💾 Checkpoint saved: batch_0020.json

📁 All checkpoints saved in: /content/drive/MyDrive/Assignment3/checkpoint_task_3_1_evaluation__section_chunks__baseline/

✅ RAGAS evaluation complete | 496 records scored


In [12]:
# ══════════════════════════════════════════════════════════
# STEP 8 — MERGE RAGAS SCORES + HIT RATE INTO PER-PAIR RECORDS
# ══════════════════════════════════════════════════════════

def safe_float(val) -> float | None:
    """CONVERT VALUE TO FLOAT, RETURNING None ON FAILURE**"""
    try:
        f = float(val)
        return round(f, 4) if not math.isnan(f) else None
    except (TypeError, ValueError):
        return None


# BUILD A LOOKUP: query_id → hit (True/False/None) **
qid_to_hit = {}
for rec, hit in zip(results, per_pair_hits):
    qid_to_hit[rec["query_id"]] = hit

# BUILD FINAL PER-PAIR RECORDS **
per_pair_results = []

score_idx = 0  # **INDEX INTO ragas_scores (ONLY VALID RECORDS WERE SCORED)**

for rec in results:
    qid      = rec["query_id"]
    answer   = rec["answer"]
    is_valid = answer.strip() not in (
        "Generation failed.",
        "Generation failed due to API error.",
        ""
    )  # **MATCH FILTER APPLIED IN build_ragas_dataset**

    if is_valid and score_idx < len(ragas_scores):
        scores  = ragas_scores[score_idx]
        score_idx += 1  # **ADVANCE RAGAS INDEX ONLY FOR VALID RECORDS**
    else:
        scores = {}  # **FAILED GENERATION — NO RAGAS SCORES**

    pair = {
        "query_id":            qid,
        "query":               rec["query"],
        "modality":            rec["modality"],
        "answer":              answer,
        "ground_truth":        rec["ground_truth"],
        "hit_at_k":            qid_to_hit.get(qid),      # **True/False/None**
        "retrieved_arxiv_ids": [
            c["arxiv_id"] for c in rec["retrieved_chunks"][:HIT_K]
        ],
        # ── RAGAS METRICS ──────────────────────────────
        "faithfulness":        safe_float(scores.get("faithfulness")),
        "answer_relevancy":    safe_float(scores.get("answer_relevancy")),
        "context_precision":   safe_float(scores.get("context_precision")),
        "context_recall":      safe_float(scores.get("context_recall")),
        "answer_correctness":  safe_float(scores.get("answer_correctness")),
    }
    per_pair_results.append(pair)

print(f"✅ Merged {len(per_pair_results)} per-pair records")
print(f"   RAGAS scored: {score_idx} | Skipped (failed gen): "
      f"{len(per_pair_results) - score_idx}")

✅ Merged 496 per-pair records
   RAGAS scored: 496 | Skipped (failed gen): 0


In [13]:
# ══════════════════════════════════════════════════════════
# STEP 9 — COMPUTE AGGREGATE METRICS
# ══════════════════════════════════════════════════════════

def aggregate_metrics(per_pair: list) -> dict:
    """
    COMPUTE MEAN ± STD FOR EACH METRIC ACROSS ALL VALID PAIRS**
    IGNORES None VALUES (FAILED GENERATIONS OR UNSCORED RECORDS)**
    """
    metric_keys = [
        "faithfulness",
        "answer_relevancy",
        "context_precision",
        "context_recall",
        "answer_correctness",
    ]
    agg = {}
    for key in metric_keys:
        vals = [p[key] for p in per_pair if p[key] is not None]  # **DROP NONES**
        if vals:
            agg[key] = {
                "mean":   round(float(np.mean(vals)), 4),
                "std":    round(float(np.std(vals)),  4),
                "median": round(float(np.median(vals)), 4),
                "min":    round(float(np.min(vals)),  4),
                "max":    round(float(np.max(vals)),  4),
                "n":      len(vals),
            }
        else:
            agg[key] = None  # **NO VALID SCORES FOR THIS METRIC**

    # HIT RATE STATS **
    hits_valid  = [p["hit_at_k"] for p in per_pair if p["hit_at_k"] is not None]
    agg[f"hit_rate_at_{HIT_K}"] = {
        "mean": round(sum(hits_valid) / len(hits_valid), 4) if hits_valid else None,
        "hits": sum(hits_valid),
        "n":    len(hits_valid),
    }

    return agg


def aggregate_by_modality(per_pair: list) -> dict:
    """
    COMPUTE MEAN METRICS GROUPED BY QUERY MODALITY**
    ENABLES COMPARISON: text-only vs multimodal QUERIES**
    """
    metric_keys = [
        "faithfulness", "answer_relevancy", "context_precision",
        "context_recall", "answer_correctness",
    ]
    grouped = defaultdict(list)
    for p in per_pair:
        grouped[p["modality"]].append(p)

    by_modality = {}
    for mod, pairs in sorted(grouped.items()):
        mod_agg = {"n": len(pairs)}
        for key in metric_keys:
            vals = [p[key] for p in pairs if p[key] is not None]
            mod_agg[key] = round(float(np.mean(vals)), 4) if vals else None
        hits = [p["hit_at_k"] for p in pairs if p["hit_at_k"] is not None]
        mod_agg[f"hit_rate_at_{HIT_K}"] = (
            round(sum(hits) / len(hits), 4) if hits else None
        )  # **HIT RATE PER MODALITY**
        by_modality[mod] = mod_agg

    return by_modality


aggregate   = aggregate_metrics(per_pair_results)
by_modality = aggregate_by_modality(per_pair_results)

print("\n" + "═"*60)
print("  AGGREGATE RAGAS METRICS (ALL QUERIES)")
print("═"*60)
metric_labels = {
    "faithfulness":        "Faithfulness",
    "answer_relevancy":    "Answer Relevancy",
    "context_precision":   "Context Precision",
    "context_recall":      "Context Recall",
    "answer_correctness":  "Answer Correctness",
    f"hit_rate_at_{HIT_K}": f"Hit Rate@{HIT_K}",
}
for key, label in metric_labels.items():
    val = aggregate.get(key)
    if val and "mean" in val:
        print(f"  {label:<22}: {val['mean']:.4f}  "
              f"(±{val.get('std', 0):.4f})  n={val['n']}")
    elif val:
        print(f"  {label:<22}: {val.get('mean', 'N/A'):.4f}  "
              f"hits={val.get('hits', '?')}/{val.get('n', '?')}")
    else:
        print(f"  {label:<22}: N/A")

print("\n" + "═"*60)
print("  METRICS BY MODALITY")
print("═"*60)
col_keys = ["faithfulness", "answer_relevancy", "context_precision",
            "context_recall", "answer_correctness", f"hit_rate_at_{HIT_K}", "n"]
header = f"  {'Modality':<22}" + "".join(f"  {k[:6]:>8}" for k in col_keys)
print(header)
print("  " + "-"*90)
for mod, stats in by_modality.items():
    row = f"  {mod:<22}"
    for k in col_keys:
        v = stats.get(k)
        row += f"  {v:>8.4f}" if isinstance(v, float) else f"  {str(v):>8}"
    print(row)


════════════════════════════════════════════════════════════
  AGGREGATE RAGAS METRICS (ALL QUERIES)
════════════════════════════════════════════════════════════
  Faithfulness          : 0.7303  (±0.3794)  n=495
  Answer Relevancy      : 0.6881  (±0.4073)  n=494
  Context Precision     : 0.8602  (±0.2761)  n=496
  Context Recall        : 0.7372  (±0.4182)  n=496
  Answer Correctness    : 0.4330  (±0.2876)  n=495
  Hit Rate@3            : 0.9677  (±0.0000)  n=496

════════════════════════════════════════════════════════════
  METRICS BY MODALITY
════════════════════════════════════════════════════════════
  Modality                  faithf    answer    contex    contex    answer    hit_ra         n
  ------------------------------------------------------------------------------------------
  text                      0.7814    0.7320    0.8860    0.7538    0.4680    0.9666       329
  text-image                0.6838    0.6536    0.8406    0.7290    0.3777    0.9826       115
  text-t

In [14]:
# ══════════════════════════════════════════════════════════
# STEP 10 — IDENTIFY WORST PERFORMERS (ERROR ANALYSIS)
# ══════════════════════════════════════════════════════════

def find_worst_pairs(per_pair: list, metric: str = "answer_correctness",
                     n: int = 5) -> list:
    """
    IDENTIFY TOP-N WORST PERFORMING Q&A PAIRS BY A CHOSEN METRIC**
    USEFUL FOR TASK 3.2 ERROR ANALYSIS IN THE WRITEUP**
    """
    scored = [
        p for p in per_pair if p.get(metric) is not None
    ]  # **DROP UNSCORED RECORDS**
    return sorted(scored, key=lambda x: x[metric])[:n]  # **ASCENDING = WORST FIRST**


def find_hallucinations(per_pair: list, threshold: float = 0.3,
                         n: int = 5) -> list:
    """
    FIND LIKELY HALLUCINATIONS: LOW FAITHFULNESS BUT HIGH ANSWER RELEVANCY**
    THESE ARE ANSWERS THAT SOUND RELEVANT BUT AREN'T GROUNDED IN CONTEXT**
    """
    return [
        p for p in per_pair
        if p.get("faithfulness") is not None
        and p["faithfulness"] < threshold
        and (p.get("answer_relevancy") or 0) > 0.5
    ][:n]  # **TOP N CANDIDATES**


worst_overall  = find_worst_pairs(per_pair_results, metric="answer_correctness", n=5)
worst_faithful = find_worst_pairs(per_pair_results, metric="faithfulness", n=5)
hallucinations = find_hallucinations(per_pair_results)
worst_multimodal = [
    p for p in per_pair_results
    if p["modality"] != "text" and p.get("answer_correctness") is not None
]
worst_multimodal = sorted(worst_multimodal, key=lambda x: x["answer_correctness"])[:3]

print("\n" + "═"*60)
print(f"  TOP-3 WORST PAIRS BY ANSWER CORRECTNESS")
print("═"*60)
for i, p in enumerate(worst_overall[:3], 1):
    print(f"\n  [{i}] query_id : {p['query_id']}")
    print(f"       modality : {p['modality']}")
    print(f"       correctness: {p['answer_correctness']} | "
          f"faithfulness: {p['faithfulness']} | "
          f"hit@{HIT_K}: {p['hit_at_k']}")
    print(f"       query    : {p['query'][:100]}")
    print(f"       answer   : {p['answer'][:150]}...")
    print(f"       gt       : {p['ground_truth'][:150]}...")

print("\n" + "═"*60)
print(f"  POTENTIAL HALLUCINATIONS (faithfulness < 0.3)")
print("═"*60)
if hallucinations:
    for i, p in enumerate(hallucinations[:3], 1):
        print(f"\n  [{i}] query_id   : {p['query_id']}")
        print(f"       modality   : {p['modality']}")
        print(f"       faithfulness: {p['faithfulness']} | "
              f"answer_relevancy: {p['answer_relevancy']}")
        print(f"       query      : {p['query'][:100]}")
        print(f"       answer     : {p['answer'][:200]}...")
else:
    print("  ✅ No hallucination candidates found at threshold 0.3")


════════════════════════════════════════════════════════════
  TOP-3 WORST PAIRS BY ANSWER CORRECTNESS
════════════════════════════════════════════════════════════

  [1] query_id : 86390a13-479f-463f-9cb0-0b668968d28a
       modality : text-image
       correctness: 0.01 | faithfulness: 0.0 | hit@3: True
       query    : What is the highest performing synthesizer for the US8K task?
       answer   : I don't have enough information in the provided context to answer this question. 

Sources Used: None...
       gt       : VoiceFM...

  [2] query_id : 85e0dff2-6473-43e5-b555-9b69c9c31564
       modality : text-table
       correctness: 0.0156 | faithfulness: 0.0 | hit@3: False
       query    : How are positive-sense single-stranded RNA viruses translated into proteins?
       answer   : I don't have enough information in the provided context to answer this question. 

Sources Used: None...
       gt       : Positive-sense single-stranded RNA viruses can be directly translated into pro

In [15]:
# ══════════════════════════════════════════════════════════
# STEP 11 — SAVE FULL RESULTS TO JSON
# ══════════════════════════════════════════════════════════

output = {
    "task":        "3_ragas_evaluation",
    "source_file": RESULTS_PATH,
    "ragas_llm":   RAGAS_LLM,
    "ragas_embed": RAGAS_EMBED,
    "hit_k":       HIT_K,
    "n_evaluated": len(per_pair_results),
    # ── AGGREGATE METRICS ─────────────────────────────────
    "aggregate":         aggregate,
    "by_modality":       by_modality,
    # ── ERROR ANALYSIS ────────────────────────────────────
    "worst_by_correctness":   worst_overall,
    "worst_by_faithfulness":  worst_faithful,
    "hallucination_candidates": hallucinations,
    "worst_multimodal":       worst_multimodal,
    # ── PER-PAIR SCORES ───────────────────────────────────
    "per_pair": per_pair_results,
}  # **FULL OUTPUT STRUCTURE**

with open(OUTPUT_PATH, "w") as f:
    json.dump(output, f, indent=2, default=str)  # **default=str HANDLES NaN/None**

print(f"✅ Results saved to {OUTPUT_PATH}")
print(f"   Total records  : {len(per_pair_results)}")
print(f"   File size      : {Path(OUTPUT_PATH).stat().st_size / 1024:.1f} KB")

✅ Results saved to /content/drive/MyDrive/Assignment3/task_3_1_evaluation__section_chunks__baseline.json
   Total records  : 496
   File size      : 542.0 KB
